<b><font size="6" color="#E8800A">Week 10a · Bagging and random forests</font></b><br>
<b><font size="4">Averaging bootstrap fits steadies an unstable learner; a forest adds column sampling inside the tree</font></b><br>

Bagging fits a learning algorithm repeatedly on resampled training rows and combines the predictions. A random forest is a bagged ensemble of trees whose splits also draw their candidate columns at random.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Bagging by hand</font>](#toy)
* [<font color='#E8800A'>Bagging and its base learner</font>](#bagging)
* [<font color='#E8800A'>Four sampling constructions</font>](#sampling)
* [<font color='#E8800A'>Out-of-bag predictions</font>](#oob)
* [<font color='#E8800A'>Bagging settings</font>](#bag_settings)
* [<font color='#E8800A'>Random forests</font>](#forest)
* [<font color='#E8800A'>Forest settings</font>](#forest_settings)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The final grid and its winner</font>](#grid)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Creating two equal models</font>](#equal)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Bagging by hand</font> <a class="anchor" id="toy"></a>
[Back to TOC](#toc)

A bagged ensemble has three parts: a bootstrap draw of the training rows for each member, one fitted tree per draw, and the average of the members' answers. Ten rows are small enough to carry all three by hand.

In [ ]:
# Make this folder importable, so the course modules load from it.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Libraries for timing, parallel runs, tables and plots.
import os
import time
import warnings
import joblib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
# The scikit-learn pieces: ensembles, base learners, metrics and the splitting tools.
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from tqdm.auto import tqdm
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import ParameterGrid, cross_validate, train_test_split, StratifiedKFold
# The course helpers hold the plot colours, the record log and the scoreboard.
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, result_context, results_before

# One seed for every random draw, so a rerun gives the same numbers.
random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
# The encoder warns when a held-out row holds a category its training rows never showed. It encodes that row as all zeros by design, so this one message is silenced.
warnings.filterwarnings("ignore", message="Found unknown categories.*during transform", category=UserWarning)
# Worker processes start fresh and ignore the filter above, so the same exact message is also silenced through the environment they inherit; no other warning is touched.
os.environ["PYTHONWARNINGS"] = "ignore:Found unknown categories:UserWarning"
# Rows that no member omitted have no out-of-bag prediction; the notebook counts them in a column, so scikit-learn's warning about them is silenced.
warnings.filterwarnings("ignore", message="Some inputs do not have OOB scores.*", category=UserWarning)
# The same rows make scikit-learn divide zero by zero when it normalises their out-of-bag probabilities; they stay NaN and are left out of the scores.
warnings.filterwarnings("ignore", message="invalid value encountered in divide", category=RuntimeWarning, module="sklearn.ensemble._bagging")
print("scikit-learn", sklearn.__version__)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


A bootstrap draw samples rows **with replacement**: a row can appear several times and another can be absent. Each member fits the base algorithm to its own draw. Regression averages the members' predictions; classification, in Breiman's original rule, takes the class most members predict. [scikit-learn bagging guide](https://scikit-learn.org/stable/modules/ensemble.html#bagging-meta-estimator)

__Step 1:__ Write down ten toy athletes: weekly training hours and whether the athlete won.

In [ ]:
toy = pd.DataFrame({"hours": [2, 4, 6, 8, 10, 12, 14, 16, 18, 20],
                    "won": [0, 0, 0, 1, 0, 1, 1, 0, 1, 1]})
toy.T

__Step 2:__ Draw three bootstrap samples of the ten row positions, one for each member.

In [ ]:
toy_generator = np.random.RandomState(random_state)
toy_draws = [toy_generator.choice(10, size=10, replace=True) for member in range(3)]
for member, rows in enumerate(toy_draws, start=1):
    print(f"member {member}: rows {rows}, {len(np.unique(rows))} distinct, {10 - len(np.unique(rows))} left out")

__Step 3:__ Count how often each row position appears in each draw.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for member, (rows, axis) in enumerate(zip(toy_draws, axes), start=1):
    sns.barplot(x=np.arange(10), y=np.bincount(rows, minlength=10), color=PLOT_ORANGE, ax=axis)
    axis.set(xlabel="Row position", title=f"Member {member}")
axes[0].set_ylabel("Times drawn")
plt.tight_layout()
plt.show()

**The three draws leave out 3, 4 and 6 of the ten rows.** Member 3 draws row 3 four times and holds 4 distinct rows, while member 1 holds 7.

With $n$ training rows and $m$ draws with replacement, one row is omitted with probability $(1-1/n)^m$. For $m=n$ the probability approaches $e^{-1}$ as $n$ grows, where $e$ is Euler's number, so a full-size bootstrap leaves out about 36.8 percent of the rows when $n$ is large, and it typically omits some rows at any size. The shares measured on the course rows in Steps 22 and 23 below show this limit at work. [Breiman, Random Forests](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf)

__Step 4:__ Fit one tree with two questions to each draw.

In [ ]:
toy_trees = [DecisionTreeClassifier(max_depth=2, random_state=random_state).fit(toy[["hours"]].iloc[rows], toy["won"].iloc[rows])
             for rows in tqdm(toy_draws, desc='toy draws', leave=True, position=0)]

__Step 5:__ Draw the three fitted trees.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for member, (tree, axis) in enumerate(zip(toy_trees, axes), start=1):
    plot_tree(tree, feature_names=["hours"], class_names=["lost", "won"], filled=True, rounded=True, precision=2, fontsize=10, ax=axis)
    axis.set_title(f"Member {member}")
plt.tight_layout()
plt.show()

Bagging takes the plurality vote of the members, the rule Breiman gave in [Bagging Predictors](https://www.stat.berkeley.edu/~breiman/bagging.pdf): $\hat y(x)=\arg\max_k \sum_{b=1}^{M}\mathbb{1}[\hat y_b(x)=k]$. Here $M$ is the number of members, $b$ indexes them, $x$ is an input row, $k$ is a class, $\hat y_b(x)$ is the class member $b$ predicts for $x$, and $\mathbb{1}[\hat y_b(x)=k]$ equals 1 when that prediction is class $k$ and 0 otherwise, so the sum counts the votes for class $k$ and $\hat y(x)$ is the class with the most votes. With three members and two classes no ties occur. In scikit-learn, `BaggingClassifier` averages the members' class probabilities when they supply them and votes only otherwise, as the [BaggingClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html) states.

__Step 6:__ Take the vote of the three members for each of the ten rows.

In [ ]:
toy_answers = pd.DataFrame({f"member {member}": tree.predict(toy[["hours"]]) for member, tree in enumerate(toy_trees, start=1)})
toy_answers.insert(0, "won", toy["won"])
toy_answers.insert(0, "hours", toy["hours"])
toy_answers["vote"] = toy_answers.filter(like="member").mode(axis=1)[0].astype(int)
toy_answers

**At 14 hours the members vote 1, 1 and 0, so two of the three carry the vote and the ensemble class is 1, the athlete's outcome.** At 10 hours all three members vote 1 for an athlete whose outcome is 0, so the vote cannot repair an error that every member makes.

# <font color='#E8800A'>Bagging and its base learner</font> <a class="anchor" id="bagging"></a>
[Back to TOC](#toc)

A deep tree has high instability: its splits can change when a few training rows change. Bagging combines several such fits, and the gain depends on how the members' errors move together; diversity alone does not make the predictions accurate.

__Step 7:__ Load the course spine and fix the outer holdout.

In [ ]:
# Keep the outer test rows separate from every training-fold comparison.
results_dir = "../../results"
frame, recipe = load_recipe("../../data/interim/champions.csv", classification_recipe_log)
frame = frame.set_index("RecordID")
features = frame.loc[:, recipe["categorical"] + recipe["numeric"]].copy()
target = frame["Outcome"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=random_state, stratify=target)
print(f"{len(X_train):,} training rows; {len(X_test):,} outer-test rows")

__Step 8:__ Prepare the training partition once, for the ten folds and for all the training rows, and fix the folds every validation score is read on.

In [ ]:
# The training partition is prepared once here: each fold's fitted and held-out rows, then all the training rows and the test rows.
folds = outer_splitter("classification", random_state=random_state)
prepared = prepared_folds(FoldPreprocessor(recipe), folds, X_train, y_train, n_jobs=4)
# The recipe fitted on all the training rows prepares them as Z_train and the test rows as Z_test, which every all-rows fit below reuses.
full_prep = FoldPreprocessor(recipe).fit(X_train, y_train)
Z_train = full_prep.transform(X_train)
Z_test = full_prep.transform(X_test)
print(f"{len(prepared)} paired training folds")

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html'>sklearn.ensemble.BaggingClassifier(estimator=None, n_estimators=10, *, max_samples=1.0, max_features=1.0, bootstrap=True, bootstrap_features=False, oob_score=False, warm_start=False, n_jobs=None, random_state=None)</a>

**Definition:**
Fits a base learner repeatedly on sampled rows and columns, then aggregates member predictions.

**Main Parameters:**
- `estimator`: learning algorithm fitted by each member.
- `n_estimators`: number of members.
- `max_samples`, `bootstrap`: row draw size and replacement.
- `max_features`, `bootstrap_features`: member column draw size and replacement.
- `oob_score`: bootstrap-only diagnostic; incompatible with warm_start.
- `random_state`, `n_jobs`, `warm_start`: repeatability, workers and incremental growth.

**Main Attributes (after fitting):**
- `estimators_`: separately fitted members.
- `estimators_samples_`: drawn row positions.
- `estimators_features_`: drawn source-column positions.
- `oob_score_`: eligible out-of-bag score after fitting.

**Main Methods:**
- `fit(X, y)`: fit members and aggregate their predictions.
- `predict(X)`: predict through the ensemble.
- `predict_proba(X)`: average member probabilities, when available.
</div>

`BaggingClassifier` chooses the learning algorithm through `estimator`; leaving it unspecified selects a decision tree. A shallow tree, a deep tree and logistic regression are different base algorithms or settings, even when the wrapper draws the same rows. [API and defaults](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)

__Step 9:__ Build the default ensemble, whose base learner is `DecisionTreeClassifier(random_state=random_state)`.

In [ ]:
default_bag = BaggingClassifier(estimator=..., n_estimators=10, random_state=random_state, n_jobs=4)

__Step 10:__ Score the default ensemble on the ten folds.

In [ ]:
# Score each fold with a fresh clone fitted only on its training rows.
default_train, default_validation = [], []
default_started = time.perf_counter()
for fold in tqdm(prepared, desc='prepared', leave=True, position=0):
    fitted = clone(default_bag).fit(fold["X_fit"], fold["y_fit"])
    default_train.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
    default_validation.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
default_seconds = time.perf_counter() - default_started
print(f"default bagging: train F1 {np.mean(default_train):.4f}, validation F1 {np.mean(default_validation):.4f}, {default_seconds:.2f} seconds for ten folds")

**The default ensemble of ten trees scores F1 0.9911 on the rows it was fitted on and 0.8374 on the validation folds.** The gap is the signature of deep trees, which memorise their fitted rows.

__Step 11:__ Fit the default ensemble on all the training rows.

In [ ]:
default_fitted = clone(default_bag).fit(Z_train, y_train)

__Step 12:__ Open the log with the default ensemble.

In [ ]:
model_log = []
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": " ".join(str(default_bag).split()), "train": default_train,
    "validation": default_validation, "model": default_fitted},
    Z_test, y_test, metric=f1_score, label="F1")

__Step 13:__ Read the ten members' answers for the one of the first thirty rows where they disagree most.

In [ ]:
# The ensemble fitted on all the training rows, and the prepared rows its members were fitted on.
bagged = default_fitted
aggregation_rows = Z_train[:30]
# Each member answers on the columns it drew; estimators_features_ lists them.
member_outputs = np.asarray([member.predict_proba(aggregation_rows[:, columns])[:, 1]
    for member, columns in zip(bagged.estimators_, bagged.estimators_features_)])
# The row where the members disagree most.
aggregation_at = int(member_outputs.std(axis=0).argmax())
individual_answers = member_outputs[:, aggregation_at]
# What the wrapper itself returns for that row.
wrapper_answer = bagged.predict_proba(aggregation_rows[[aggregation_at]])[:, 1][0]
# The members' answers, their mean and the wrapper's answer; the assert requires the mean and the wrapper to agree.
print("member answers:", individual_answers.round(4))
print("arithmetic mean:", individual_answers.mean())
print("wrapper answer:", wrapper_answer)
assert np.isclose(individual_answers.mean(), wrapper_answer)

__Step 14:__ Plot the ten answers beside their mean.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.scatterplot(x=np.arange(1, len(individual_answers) + 1), y=individual_answers, color=PLOT_BLUE, ax=ax)
ax.axhline(individual_answers.mean(), color=PLOT_ORANGE, label="ensemble mean")
ax.set(xlabel="Member", ylabel="Positive-class probability", title="One training row, ten member answers")
ax.legend()
plt.tight_layout()
plt.show()

**Of the ten members, 3 answer 1 and 7 answer 0 for this row, and the mean 0.3 equals the wrapper's answer.** The wrapper's `predict_proba` is this arithmetic mean and `predict` takes the larger class, which is the probability averaging described in the documentation linked above.

Each learner below is the configuration stored on the course board, beside the unpruned deep tree, whose splits change most when the rows change. Fitting each alone and bagged on the same folds shows which of them gain from averaging. [Bagging and unstable learners](https://www.stat.berkeley.edu/~breiman/bagging.pdf)

__Step 15:__ Write the base learners once, each in the configuration stored on the board, beside the unpruned deep tree.

In [ ]:
# Reuse the configurations already measured in the course benchmark.
taught_learners = {
    "deep tree": DecisionTreeClassifier(random_state=random_state),
    "logistic regression": LogisticRegression(solver="saga", l1_ratio=1.0, C=100, fit_intercept=False, max_iter=5000),
    "naive Bayes": CalibratedClassifierCV(GaussianNB(var_smoothing=0.1, priors=None), method="isotonic"),
    "KNN": KNeighborsClassifier(n_neighbors=41, p=1, weights="distance"),
    "pruned tree": DecisionTreeClassifier(ccp_alpha=0.001, class_weight=None, criterion="gini", max_depth=None,
                                          max_features=None, min_samples_leaf=20, min_samples_split=50,
                                          splitter="best", random_state=random_state),
}

__Step 16:__ Print each constructor beside its stored board label.

In [ ]:
for (name, model), week in zip(taught_learners.items(), [None, 6, 7, 8, 9]):
    stored = "none, the unpruned default tree" if week is None else json.loads(
        Path(results_dir, f"week_{week:02d}.json").read_text(encoding="utf-8"))["classification"]["model"]
    print(name, "| stored label:", stored)
    print("   constructor:", " ".join(str(model).split()))

**Each constructor carries the settings of its stored label.** The tree constructors add `random_state=33`, which the stored labels omit, and the naive Bayes label's isotonic calibration appears as `CalibratedClassifierCV(..., method='isotonic')`.

Inside a bagging member the naive Bayes learner receives the rows the fold already prepared, so its internal calibration splits share that preparation.

__Step 17:__ Fit each learner alone and bagged on every fold, timing both, and measure how much the bagged members disagree.

In [ ]:
# One row per learner and fold: the learner alone and bagged, their fit times and the spread of the bagged members.
base_rows = []
member_variation = []
for learner_name, learner in tqdm(taught_learners.items(), desc='taught learners', leave=True, position=0):
    for fold_id, fold in enumerate(tqdm(prepared, desc=learner_name, unit="fold", leave=False)):
        # Fit the learner alone, then bagged with ten members, on the same fold.
        started = time.perf_counter()
        single = clone(learner).fit(fold["X_fit"], fold["y_fit"])
        single_seconds = time.perf_counter() - started
        started = time.perf_counter()
        bagged_fold = BaggingClassifier(estimator=clone(learner), n_estimators=10,
            random_state=random_state, n_jobs=4).fit(fold["X_fit"], fold["y_fit"])
        bagged_seconds = time.perf_counter() - started
        # Each bagged member's answer on the validation rows; the spread across members measures sensitivity to the bootstrap.
        member_answers = np.asarray([member.predict_proba(fold["X_score"][:, columns])[:, 1]
            for member, columns in zip(bagged_fold.estimators_, bagged_fold.estimators_features_)])
        # Keep the scores and the times of this learner and fold.
        base_rows.append(dict(learner=learner_name, fold=fold_id,
            single=f1_score(fold["y_score"], single.predict(fold["X_score"])),
            bagged=f1_score(fold["y_score"], bagged_fold.predict(fold["X_score"])),
            single_seconds=single_seconds, bagged_seconds=bagged_seconds))
        # Keep the mean spread of the members for this learner and fold.
        member_variation.append(dict(learner=learner_name, fold=fold_id,
            **{"member prediction SD": member_answers.std(axis=0).mean()}))
base_comparison = pd.DataFrame(base_rows)

__Step 18:__ Average the scores over the ten folds, and add up the fitting time.

In [ ]:
display(base_comparison.groupby("learner", sort=False)[["single", "bagged"]].mean().round(4))
display(pd.DataFrame(member_variation).groupby("learner", sort=False)["member prediction SD"].mean().round(4))
display(base_comparison.groupby("learner", sort=False)[["single_seconds", "bagged_seconds"]].sum().round(2))

The member standard deviation measures sensitivity to the bootstrap on the same validation rows. It is expressed in positive-class probability units and is not an error score. The single and bagged columns measure predictive performance on identical folds.

**Bagging lifts the deep tree's F1 from 0.7996 to 0.8374 and the pruned tree's from 0.8543 to 0.8590, while logistic regression moves from 0.8511 to 0.8517, naive Bayes from 0.8047 to 0.8036 and KNN from 0.8522 to 0.8534.** The two trees disagree most between draws, 0.2782 for the deep tree and 0.1306 for the pruned tree, and the deep tree gains most; logistic regression, naive Bayes and KNN disagree less, 0.0339, 0.0266 and 0.0582, and gain almost nothing. Bagging logistic regression is by far the slowest, 26.35 seconds in total against 0.45 for the deep tree.

__Step 19:__ Plot the fold scores and the member disagreement of every learner.

In [ ]:
# Left: validation scores of each learner alone and bagged; right: how far the bagged members disagree.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
# One long table, so the box plot can colour alone against bagged.
base_long = base_comparison.melt(id_vars=["learner", "fold"], value_vars=["single", "bagged"], var_name="fit", value_name="validation score")
sns.boxplot(data=base_long, x="learner", y="validation score", hue="fit", palette=[PLOT_BLUE, PLOT_ORANGE], ax=axes[0])
sns.barplot(data=pd.DataFrame(member_variation), x="learner", y="member prediction SD", errorbar=None, color=PLOT_BLUE, ax=axes[1])
axes[0].set_title("Matched validation scores")
axes[1].set_title("Sensitivity to bootstrap resampling")
axes[0].set_ylabel("Validation F1")
axes[1].set_ylabel("Member prediction SD (probability)")
# Tilt the learner names so they stay readable.
for axis in axes:
    axis.tick_params(axis="x", labelrotation=20)
plt.tight_layout()
plt.show()

**The bagged deep tree's box is shifted above the single tree's, in line with the rise in the mean printed above, while for logistic regression, naive Bayes and KNN the single and bagged boxes overlap almost completely.**

# <font color='#E8800A'>Four sampling constructions</font> <a class="anchor" id="sampling"></a>
[Back to TOC](#toc)

The base algorithm and the sampling construction are separate choices. The same bagging class produces the four named constructions below; they differ in how rows and columns reach each member, not in which base learner is fitted.

| Construction | `max_samples` | `bootstrap` | `max_features` | `bootstrap_features` |
| --- | --- | --- | --- | --- |
| Pasting | 0.7 | False | 1.0 | False |
| Bagging | 1.0 | True | 1.0 | False |
| Random subspaces | 1.0 | False | 0.7 | False |
| Random patches | 0.7 | False | 0.7 | False |

These are representative configurations of the named methods, not four distinct estimators. [scikit-learn taxonomy and references](https://scikit-learn.org/stable/modules/ensemble.html#bagging-meta-estimator)

__Step 20:__ Fit a bagged tree ensemble under each construction on every fold.

In [ ]:
# The four constructions as settings of one bagging class.
sampling_methods = {
    "pasting": dict(max_samples=0.7, bootstrap=False, max_features=1.0, bootstrap_features=False),
    "bagging": dict(max_samples=1.0, bootstrap=True, max_features=1.0, bootstrap_features=False),
    "random subspaces": dict(max_samples=1.0, bootstrap=False, max_features=0.7, bootstrap_features=False),
    "random patches": dict(max_samples=0.7, bootstrap=False, max_features=0.7, bootstrap_features=False),
}
# Fit every construction on every fold, and keep the first fold's ensembles to inspect their draws.
sampling_rows = []
first_fold_models = {}
for method, settings in tqdm(sampling_methods.items(), desc='sampling methods', leave=True, position=0):
    for fold_id, fold in enumerate(prepared):
        fitted = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=random_state),
            n_estimators=10, random_state=random_state, n_jobs=4, **settings).fit(fold["X_fit"], fold["y_fit"])
        sampling_rows.append(dict(method=method, fold=fold_id,
            validation=f1_score(fold["y_score"], fitted.predict(fold["X_score"]))))
        if fold_id == 0:
            first_fold_models[method] = fitted
sampling_table = pd.DataFrame(sampling_rows)

__Step 21:__ Average the validation score of each construction over the ten folds.

In [ ]:
sampling_table.groupby("method")["validation"].mean().round(4)

**Bagging scores F1 0.8374, pasting 0.8359, random patches 0.8367 and random subspaces 0.8327.** The four constructions use one base tree and the same folds, and the bootstrap draw scores highest here, by a small margin.

__Step 22:__ Count the rows and columns the first member of each construction drew on the first fold.

In [ ]:
for method, fitted in first_fold_models.items():
    rows, columns = fitted.estimators_samples_[0], fitted.estimators_features_[0]
    print(f"{method:>16}: {len(rows):,} rows drawn, {len(np.unique(rows)):,} distinct, {len(columns)} columns drawn")
print(f"share omitted by a full-size bootstrap: {1 - len(np.unique(first_fold_models['bagging'].estimators_samples_[0])) / len(first_fold_models['bagging'].estimators_samples_[0]):.4f}, e^-1 = {np.exp(-1):.4f}")

**A bagging member draws 2,560 rows, as many as one fold fits on, of which 1,632 are distinct, so 0.3625 of the draw is left out, against the limit 0.3679.** Pasting draws 1,792 rows and all 1,792 are distinct, and the two column constructions draw 19 of the 28 columns.

`max_samples` controls the row draw size; with replacement it counts draws, not distinct rows. Bagging `max_features` selects one input-column subset **per estimator**, reused by every node of that member. With `bootstrap_features=True`, columns are sampled with replacement, so the same source column can occupy more than one input position. [Bagging sampling parameters](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)

# <font color='#E8800A'>Out-of-bag predictions</font> <a class="anchor" id="oob"></a>
[Back to TOC](#toc)

A row is out of bag for one member when that member's bootstrap draw omitted it. Its out-of-bag prediction aggregates only the members that omitted it, not the whole ensemble, and a row that every member drew has no out-of-bag prediction. The omission rate of about 36.8 percent is per member.

__Step 23:__ Count, for each training row, how many of the ten default members omitted it.

In [ ]:
member_omits = np.zeros(len(y_train), dtype=int)
for rows in bagged.estimators_samples_:
    member_omits += np.bincount(rows, minlength=len(y_train)) == 0
omit_distribution = pd.Series(member_omits).value_counts().reindex(range(len(bagged.estimators_) + 1), fill_value=0)
print("rows by number of members that omit them:")
print(omit_distribution.rename_axis("members").rename("rows").to_string())
print(f"average share of the members that omit a row: {member_omits.mean() / len(bagged.estimators_):.4f}")

**Of the 3,200 training rows, 40 are omitted by none of the ten members and so have no out-of-bag prediction, while most rows are omitted by between 2 and 5 members.** The count peaks at 838 rows omitted by 3 members, and on average a member omits 0.3650 of the rows, the per-member rate.

The out-of-bag score is computed on the training rows: each row is predicted by the members that omitted it. The default score is accuracy, whereas this notebook ranks candidates by F1. Out-of-bag scoring needs `bootstrap=True` and cannot be combined with `warm_start=True`. [OOB API and eligibility](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)

The sweeps below read out-of-bag error from the ensemble each setting fits on all the training rows, wherever `bootstrap=True` allows `oob_score=True`. `oob_error` is one minus the accuracy that `oob_score_` reports, and `oob_F1_error` is one minus the F1 of the class the out-of-bag probabilities select. `no_oob_rows` counts the rows that no member omitted: they have no out-of-bag prediction, `oob_score_` scores them as class 0, and the F1 leaves them out. Scikit-learn warns about such rows, and the notebook silences that warning because the count is printed. With a full-size draw without replacement, every member sees every row and no row is omitted. Smaller draws with `max_samples < 1.0` give pasting and can omit rows, but the bagging API still requires `bootstrap=True` for `oob_score=True`; the three out-of-bag columns hold NaN when `bootstrap=False`.

# <font color='#E8800A'>Bagging settings</font> <a class="anchor" id="bag_settings"></a>
[Back to TOC](#toc)

Each sweep changes one bagging setting from the default ensemble and scores every value on the same ten prepared training folds. The family summary picks the best configuration actually evaluated; interactions need the later joint search.

| Bagging parameter | Meaning and evaluation role |
| --- | --- |
| `estimator` | Base algorithm and its settings: a tree or the linear learner. |
| `n_estimators` | Number of members; score and fit time are measured together. |
| `max_samples` | Number or fraction of row draws per member. |
| `max_features` | Number or fraction of columns selected once per estimator. |
| `bootstrap` | Sample rows with replacement when True, without replacement when False. |
| `bootstrap_features` | Sample columns with replacement when True. |

[Bagging parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)

__Step 24:__ Change only bagging `estimator` on the fixed training folds, using the five learners of `taught_learners`.

### `bagging: estimator`

The `estimator` is the learning algorithm every member fits to its own draw; its default is a `DecisionTreeClassifier` grown without limits. Bagging helps most when the member is unstable, so that different draws give different fits whose errors partly cancel, and it adds little to a stable member that every draw fits almost alike. Swapping the member changes its fit to the training rows, how far the members disagree and the fitting cost together.

In [ ]:
# The values tested one at a time, with every other setting at its default.
bag_estimator_values = list(taught_learners.values())
bag_estimator_rows = []
# Each taught learner takes the place of the base tree, with every other setting at its default.
for name, value in tqdm(list(zip(taught_learners, bag_estimator_values)), desc='bag estimator values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_bag).set_params(estimator=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": name,
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    bag_estimator_rows.append(row)
# The sweep table, one row per tested value.
bag_estimator_sweep = pd.DataFrame(bag_estimator_rows)
bag_estimator_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_estimator_sweep.loc[bag_estimator_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The pruned tree wins at validation F1 0.8590, ahead of KNN's 0.8534, logistic regression's 0.8517, the deep tree's 0.8374 and naive Bayes at 0.8036.** Naive Bayes also has the highest out-of-bag error on both measures, 0.2609 for 1 - OOB accuracy and 0.1980 for 1 - OOB F1, and the deep tree is next, 0.2278 and 0.1881. Each of the five ensembles leaves 40 training rows without an out-of-bag prediction.

__Step 25:__ Change only bagging `n_estimators` on the fixed training folds, over every count from 1 to 150 members.

### `bagging: n_estimators`

`n_estimators` is the number of members averaged into one answer; its default is `10`. More members average away more of the variance between draws, with diminishing returns, while the fit to the training rows changes little and the cost grows with the count. Each fold grows one ensemble from one member to 150 with `warm_start=True`, adding one member at a time. `BaggingClassifier` refuses `oob_score=True` together with `warm_start=True`, so its out-of-bag error is accumulated through `estimators_samples_` and `estimators_features_` from one growing ensemble on all the training rows.

__Step 26:__ Score every count from 1 to 150 on the ten folds by growing one ensemble per fold, and read the test score and the out-of-bag error from one ensemble grown on all the training rows.

In [ ]:
# Every member count from 1 to 150, from one growing ensemble per fold instead of 150 separate fits.
bag_n_estimators_values = list(range(1, 151))
bag_n_estimators_rows = []
# Score tables with one row per count and one column per fold.
fit_scores = np.zeros((len(bag_n_estimators_values), len(prepared)))
validation_scores = np.zeros_like(fit_scores)
# Fit time, added up over the folds and the all-rows fit.
seconds = np.zeros(len(bag_n_estimators_values))
# Fold loop: grow one ensemble on the fitted rows of this fold and score it at every count.
for fold_id, fold in tqdm(enumerate(prepared), desc='fold id, fold', leave=True, position=0):
    # warm_start lets n_estimators rise by one with every fit and keeps the members already fitted.
    growing = clone(default_bag).set_params(warm_start=True)
    # Running sums of the member answers on the fitted rows and on the held-out rows; dividing by the count gives the ensemble's answer.
    fit_sum = np.zeros((len(fold["y_fit"]), 2))
    score_sum = np.zeros((len(fold["y_score"]), 2))
    # Count loop inside the fold: one new member and one score per count.
    for position, value in enumerate(bag_n_estimators_values):
        started = time.perf_counter()
        growing.set_params(n_estimators=value).fit(fold["X_fit"], fold["y_fit"])
        seconds[position] += time.perf_counter() - started
        # The newest member and the columns it drew.
        member, columns = growing.estimators_[-1], growing.estimators_features_[-1]
        fit_sum += member.predict_proba(fold["X_fit"][:, columns])
        score_sum += member.predict_proba(fold["X_score"][:, columns])
        # Score the ensemble at this count from the running sums.
        fit_scores[position, fold_id] = f1_score(fold["y_fit"], fit_sum.argmax(axis=1))
        validation_scores[position, fold_id] = f1_score(fold["y_score"], score_sum.argmax(axis=1))
# A second pass on all the training rows, prepared once in Step 8, gives the test score and the out-of-bag error at every count; the test rows only record a score and never choose a count.
growing = clone(default_bag).set_params(warm_start=True)
# Running sum of the member answers on the test rows.
test_sum = np.zeros((len(y_test), 2))
test_scores, oob_error, oob_f1_error, no_oob = [], [], [], []
# Out-of-bag sums: each row collects answers only from the members that did not draw it.
oob_sum = np.zeros((len(y_train), 2))
oob_labels = y_train.to_numpy()
# Count loop on all the training rows.
for position, value in tqdm(enumerate(bag_n_estimators_values), desc='position, value', leave=True, position=0):
    started = time.perf_counter()
    growing.set_params(n_estimators=value).fit(Z_train, y_train)
    seconds[position] += time.perf_counter() - started
    # estimators_samples_ gives the rows each member drew, so the rows it omitted are its out-of-bag rows.
    member, columns, rows = growing.estimators_[-1], growing.estimators_features_[-1], growing.estimators_samples_[-1]
    out = np.bincount(rows, minlength=len(y_train)) == 0
    # Add the member's answer on its out-of-bag rows only.
    oob_sum[out] += member.predict_proba(Z_train[out][:, columns])
    test_sum += member.predict_proba(Z_test[:, columns])
    # Rows with at least one out-of-bag answer; the others are left out of the F1 or MAE.
    scored = oob_sum.sum(axis=1) > 0
    test_scores.append(f1_score(y_test, test_sum.argmax(axis=1)))
    oob_error.append(1 - np.mean(oob_sum.argmax(axis=1) == oob_labels))
    oob_f1_error.append(1 - f1_score(oob_labels[scored], oob_sum[scored].argmax(axis=1)))
    no_oob.append(int((~scored).sum()))
# Time to reach each count.
seconds = np.cumsum(seconds)
# Assemble the sweep table, one row per count.
for position, value in enumerate(bag_n_estimators_values):
    model = clone(default_bag).set_params(n_estimators=value)
    bag_n_estimators_rows.append({
        "configuration": " ".join(str(model).split()), "setting": str(value),
        "train": list(fit_scores[position]), "validation": list(validation_scores[position]),
        "train_F1": fit_scores[position].mean(), "cv_F1": validation_scores[position].mean(),
        "test_F1": test_scores[position],
        "oob_error": oob_error[position], "oob_F1_error": oob_f1_error[position], "no_oob_rows": no_oob[position],
        "seconds": seconds[position]})
bag_n_estimators_sweep = pd.DataFrame(bag_n_estimators_rows)
# Show selected counts and the best one; the figure below draws all 150.
shown = sorted({1, 2, 5, 10, 25, 50, 100, 150, int(bag_n_estimators_sweep.loc[bag_n_estimators_sweep["cv_F1"].idxmax(), "setting"])})
bag_n_estimators_sweep.set_index("setting").loc[[str(count) for count in shown], ["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_n_estimators_sweep.loc[bag_n_estimators_sweep["cv_F1"].idxmax()]
best_model = clone(default_bag).set_params(n_estimators=int(best["setting"])).fit(Z_train, y_train)
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best_model},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 27:__ Plot training and validation F1 beside the out-of-bag errors against the number of bagging members.

In [ ]:
# One panel for the training and validation scores and one panel per out-of-bag error, against the number of members.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(bag_n_estimators_values, bag_n_estimators_sweep["train_F1"], linewidth=2, color=PLOT_ORANGE, label="training (ten folds)")
axes[0].plot(bag_n_estimators_values, bag_n_estimators_sweep["cv_F1"], linewidth=2, color=PLOT_BLUE, label="validation (ten folds)")
axes[1].plot(bag_n_estimators_values, bag_n_estimators_sweep["oob_error"], linewidth=2, color=PLOT_BLUE, label="1 - OOB accuracy")
axes[1].plot(bag_n_estimators_values, bag_n_estimators_sweep["oob_F1_error"], linewidth=2, linestyle="--", color=PLOT_ORANGE, label="1 - OOB F1")
axes[0].set(xlabel="number of members", ylabel="F1", title="Training and validation F1")
axes[1].set(xlabel="number of members", ylabel="out-of-bag error", title="Out-of-bag error")
# Name the lines of every panel.
for axis in axes:
    axis.legend()
plt.tight_layout()
plt.show()

**Validation F1 rises from 0.7872 with one member to 0.8374 at 10 and 0.8607 at 50, and the best of the 150 counts is 139 members at 0.8644.** Two members score lowest at 0.7302, because a one-one split averages to 0.5 and the argmax returns the negative class. Above 50 members the curve is nearly flat, 0.8611 at 100 and 0.8635 at 150, and training F1 reaches 1.0000. 1 - OOB accuracy falls from 0.4753 with one member to 0.2278 at 10 and 0.1750 at 150, and 1 - OOB F1 from 0.2149 to 0.1881 and 0.1385. With one member 2,029 training rows have no out-of-bag prediction and `oob_score_` scores them as class 0, which is why 1 - OOB accuracy starts so far above 1 - OOB F1; from 25 members none lacks one.

__Step 28:__ Change only bagging `max_samples` on the fixed training folds, with the sweep list `[0.1, 0.3, 0.5, 0.7, 1.0]`.

### `bagging: max_samples`

`max_samples` is the number or fraction of rows each member draws; its default is `1.0`, a draw as large as the training rows. Smaller draws make each member see less data and make the members differ more, which lowers variance between the averaged answers at the price of a worse fit for each member, and they cut the cost of each fit. Because rows are drawn with replacement, the number of distinct rows is smaller than the draw.

In [ ]:
# The values tested one at a time, with every other setting at its default.
bag_max_samples_values = ...
bag_max_samples_rows = []
for value in tqdm(bag_max_samples_values, desc='bag max samples values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_bag).set_params(max_samples=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    bag_max_samples_rows.append(row)
# The sweep table, one row per tested value.
bag_max_samples_sweep = pd.DataFrame(bag_max_samples_rows)
bag_max_samples_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_max_samples_sweep.loc[bag_max_samples_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8304 at 10 percent of the rows, 0.8433 at 30, 0.8410 at 50, 0.8462 at 70 and 0.8374 for the full draw.** Training F1 climbs from 0.8591 to 0.9911 over the same list. Both out-of-bag errors fall as the draw shrinks, 1 - OOB accuracy from 0.2278 for the full draw to 0.1922 at 30 percent and 1 - OOB F1 from 0.1881 to 0.1548, and both turn up again at 10 percent, 0.1966 and 0.1574, so the out-of-bag errors prefer 30 percent where validation prefers 70.

__Step 29:__ Change only bagging `max_features` on the fixed training folds, with the sweep list `[0.3, 0.5, 0.7, 1.0]`.

### `bagging: max_features`

`max_features` is the number or fraction of columns each member sees, drawn once per member and kept for all its splits; its default is `1.0`, every column. Fewer columns make the members more different and cheaper to fit, but each one has less to learn from, so the fit to the training rows loosens and the bias of the average can rise. More columns do the opposite.

In [ ]:
# The values tested one at a time, with every other setting at its default.
bag_max_features_values = [0.3, 0.5, 0.7, 1.0]
bag_max_features_rows = []
for value in tqdm(bag_max_features_values, desc='bag max features values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_bag).set_params(max_features=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    bag_max_features_rows.append(row)
# The sweep table, one row per tested value.
bag_max_features_sweep = pd.DataFrame(bag_max_features_rows)
bag_max_features_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_max_features_sweep.loc[bag_max_features_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8227 at 30 percent of the columns, 0.8321 at 50, 0.8415 at 70 and 0.8374 at all of them.** The out-of-bag errors keep falling to all the columns, 1 - OOB accuracy 0.2819, 0.2681, 0.2325 and 0.2278 and 1 - OOB F1 0.2323, 0.2229, 0.1927 and 0.1881, so they prefer all the columns where validation prefers 70 percent.

__Step 30:__ Change only bagging `bootstrap` on the fixed training folds, with the sweep list `[False, True]`.

### `bagging: bootstrap`

`bootstrap` decides whether each member draws its rows with replacement; its default is `True`. With replacement a draw repeats some rows and omits others, and that omission makes the members differ and leaves out-of-bag rows to score. Without replacement a full-size draw is the whole training set, so every member sees the same rows, the members become close to copies and no row is out of bag. With `max_samples < 1.0`, a smaller draw without replacement is pasting: members can see different subsets and omit rows, although the bagging API does not support `oob_score=True` without replacement.

In [ ]:
# The values tested one at a time, with every other setting at its default.
bag_bootstrap_values = [False, True]
bag_bootstrap_rows = []
for value in tqdm(bag_bootstrap_values, desc='bag bootstrap values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_bag).set_params(bootstrap=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    bag_bootstrap_rows.append(row)
# The sweep table, one row per tested value.
bag_bootstrap_sweep = pd.DataFrame(bag_bootstrap_rows)
bag_bootstrap_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_bootstrap_sweep.loc[bag_bootstrap_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Without replacement the validation F1 is 0.7971 with training F1 1.0000, against 0.8374 with replacement.** A full-size draw without replacement contains every row, so the members are close to copies of one tree and no row is out of bag, which leaves the out-of-bag columns NaN; with replacement 1 - OOB accuracy is 0.2278 and 1 - OOB F1 0.1881. Smaller draws without replacement (`max_samples < 1.0`) are pasting and can omit rows, but the bagging API still does not support `oob_score=True` when `bootstrap=False`.

__Step 31:__ Change only bagging `bootstrap_features` on the fixed training folds, with the sweep list `[False, True]`.

### `bagging: bootstrap_features`

`bootstrap_features` decides whether each member draws its columns with replacement; its default is `False`. With replacement a member may see one column twice and miss others, which adds variety between the members at the price of a smaller set of distinct columns for each. The fitting cost stays about the same.

In [ ]:
# The values tested one at a time, with every other setting at its default.
bag_bootstrap_features_values = [False, True]
bag_bootstrap_features_rows = []
for value in tqdm(bag_bootstrap_features_values, desc='bag bootstrap features values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_bag).set_params(bootstrap_features=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    bag_bootstrap_features_rows.append(row)
# The sweep table, one row per tested value.
bag_bootstrap_features_sweep = pd.DataFrame(bag_bootstrap_features_rows)
bag_bootstrap_features_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = bag_bootstrap_features_sweep.loc[bag_bootstrap_features_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Drawing columns with replacement scores 0.8441 against 0.8374, and its out-of-bag errors are higher, 0.2441 and 0.2032 against 0.2278 and 0.1881.** The two measures disagree here: validation prefers the repeated columns and the out-of-bag errors prefer the plain ones.

__Step 32:__ Plot training and validation F1 and the out-of-bag errors for each of the other bagging settings, one row of panels per setting.

In [ ]:
# The sweep tables of the other settings, one row of panels per setting.
bagging_sweeps = {"estimator": bag_estimator_sweep, "max_samples": bag_max_samples_sweep, "max_features": bag_max_features_sweep, "bootstrap": bag_bootstrap_sweep, "bootstrap_features": bag_bootstrap_features_sweep}
# Numeric settings get connecting lines, categories get separate points.
ordered = ("max_samples", "max_features")
# Left: training and validation score; the other panels: the out-of-bag errors.
fig, axes = plt.subplots(len(bagging_sweeps), 2, figsize=(12, 3.2 * len(bagging_sweeps)))
# One row of panels per setting, drawn from its sweep table.
for (setting, sweep), (score_axis, oob_axis) in zip(bagging_sweeps.items(), axes):
    line = "-" if setting in ordered else "none"
    # Left panel: the fold-mean training and validation scores for each value of this setting.
    score_axis.plot(sweep["setting"], sweep["train_F1"], marker="o", markersize=8, linestyle=line, color=PLOT_ORANGE, label="training (ten folds)")
    score_axis.plot(sweep["setting"], sweep["cv_F1"], marker="o", markersize=8, linestyle=line, color=PLOT_BLUE, label="validation (ten folds)")
    # Other panels: the out-of-bag errors; a value without a bootstrap has none and leaves a gap.
    oob_axis.plot(sweep["setting"], sweep["oob_error"], marker="o", markersize=8, linestyle=line, color=PLOT_BLUE, label="1 - OOB accuracy")
    oob_axis.plot(sweep["setting"], sweep["oob_F1_error"], marker="s", markersize=8, linestyle=line, color=PLOT_ORANGE, label="1 - OOB F1")
    # Titles and axis names carry the setting.
    score_axis.set(xlabel=setting, ylabel="F1", title=f"{setting}: training and validation F1")
    oob_axis.set(xlabel=setting, ylabel="out-of-bag error", title=f"{setting}: out-of-bag error")
    score_axis.legend(fontsize=8)
    oob_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

**For `max_samples`, `max_features` and `bootstrap_features` the out-of-bag errors and the validation F1 pick different values.** For `max_samples` validation is best at 70 percent of the rows, 0.8462, and the out-of-bag errors at 30 percent, 0.1922 and 0.1548. For `max_features` validation is best at 70 percent of the columns, 0.8415, and the out-of-bag errors at all of them, 0.2278 and 0.1881. For `bootstrap_features` validation prefers the repeated columns, 0.8441, and the out-of-bag errors the plain ones, 0.2278 and 0.1881. For the learner both put naive Bayes last, at validation F1 0.8036 and 1 - OOB accuracy 0.2609. The `bootstrap` panels hold one out-of-bag point, because the full-size draw without replacement omits no rows.

__Step 33:__ Select the best tested bagging configuration from the existing sweep rows by validation score.

In [ ]:
bagging_tested = pd.concat([bag_estimator_sweep, bag_n_estimators_sweep, bag_max_samples_sweep, bag_max_features_sweep, bag_bootstrap_sweep, bag_bootstrap_features_sweep], ignore_index=True)
best_tested_bagging = bagging_tested.loc[bagging_tested["cv_F1"].idxmax()]
print("best tested bagging configuration:", best_tested_bagging["configuration"])
print(best_tested_bagging[["train_F1", "cv_F1", "test_F1"]])

**The best tested bagging configuration is the deep tree with 139 members, at validation F1 0.8644 and test F1 0.8770.** It comes from the `n_estimators` sweep and beats the pruned tree's 0.8590 from the estimator sweep, so the member count helped more than any other single change.

# <font color='#E8800A'>Random forests</font> <a class="anchor" id="forest"></a>
[Back to TOC](#toc)

A random forest combines decision-tree members, bootstrap rows by default, and a fresh sample of candidate columns at each node. The row sampling resembles bagging, but the column sampling acts inside the tree.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html'>sklearn.ensemble.RandomForestClassifier(n_estimators=100, *, criterion='gini', max_depth=None, min_samples_leaf=1, max_features=sqrt, bootstrap=True, max_samples=None, random_state=None, n_jobs=None)</a>

**Definition:**
Fits decision trees with bootstrap row sampling and candidate-column sampling at each split, then aggregates their predictions.

**Main Parameters:**
- `n_estimators`: number of trees.
- `criterion`, `max_depth`, `min_samples_leaf`: tree growth controls.
- `max_features`: candidate columns sampled anew per split.
- `bootstrap`, `max_samples`: row replacement and bootstrap size.
- `random_state`, `n_jobs`: repeatability and workers.

**Main Attributes (after fitting):**
- `estimators_`: fitted trees.
- `estimators_samples_`: bootstrap row draws.

**Main Methods:**
- `fit(X, y)`: fit tree members.
- `predict(X)`: aggregate their predictions.
- `predict_proba(X)`: average tree probabilities.
</div>

Forest `max_features` controls candidate columns **per split**, whereas the bagging wrapper selects columns once per estimator, so a forest split can inspect a different set from its parent. Classification defaults to `max_features='sqrt'` and regression to `max_features=1.0`; setting all features is valid for both. The trees also expose split criterion, depth and leaf-size controls. [Random forests](https://scikit-learn.org/stable/modules/ensemble.html#forest)

__Step 34:__ Score `RandomForestClassifier(n_estimators=100)` on the ten folds.

In [ ]:
default_forest = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=4)
forest_train, forest_validation = [], []
for fold in tqdm(prepared, desc='prepared', leave=True, position=0):
    fitted = clone(default_forest).fit(fold["X_fit"], fold["y_fit"])
    forest_train.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
    forest_validation.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
print(f"forest: train F1 {np.mean(forest_train):.4f}, validation F1 {np.mean(forest_validation):.4f}")

**The default forest of 100 trees scores F1 1.0000 on its fitted rows and 0.8672 on validation, above the default bagged trees' 0.8374.** The main difference between the two is the fresh column sample at each split, together with ten times the members; the forest default takes the square root of the columns.

__Step 35:__ Fit the default forest on all the training rows.

In [ ]:
forest_fitted = clone(default_forest).fit(Z_train, y_train)

__Step 36:__ Log the default forest.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": " ".join(str(default_forest).split()), "train": forest_train,
    "validation": forest_validation, "model": forest_fitted},
    Z_test, y_test, metric=f1_score, label="F1")

# <font color='#E8800A'>Forest settings</font> <a class="anchor" id="forest_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting from the default forest.

__Step 37:__ Change only forest `n_estimators` on the fixed training folds, over every count from 1 to 150 members.

### `forest: n_estimators`

`n_estimators` is the number of trees averaged into one answer; its default is `100`. More trees average away more of the variance between draws, with diminishing returns, while the fit to the training rows changes little and the cost grows with the count. Each fold grows one forest from one tree to 150 with `warm_start=True`, adding one tree at a time. `RandomForestClassifier` accepts `warm_start=True` together with `oob_score=True`, so the forest grown on all the training rows reports its out-of-bag score after every tree.

__Step 38:__ Score every count from 1 to 150 on the ten folds by growing one forest per fold, and read the test score and the out-of-bag error from one forest grown on all the training rows.

In [ ]:
# Every member count from 1 to 150, from one growing ensemble per fold instead of 150 separate fits.
forest_n_estimators_values = list(range(1, 151))
forest_n_estimators_rows = []
# Score tables with one row per count and one column per fold.
fit_scores = np.zeros((len(forest_n_estimators_values), len(prepared)))
validation_scores = np.zeros_like(fit_scores)
# Fit time, added up over the folds and the all-rows fit.
seconds = np.zeros(len(forest_n_estimators_values))
# Fold loop: grow one ensemble on the fitted rows of this fold and score it at every count.
for fold_id, fold in tqdm(enumerate(prepared), desc='fold id, fold', leave=True, position=0):
    # warm_start lets n_estimators rise by one with every fit and keeps the members already fitted.
    growing = clone(default_forest).set_params(warm_start=True)
    # Running sums of the member answers on the fitted rows and on the held-out rows; dividing by the count gives the ensemble's answer.
    fit_sum = np.zeros((len(fold["y_fit"]), 2))
    score_sum = np.zeros((len(fold["y_score"]), 2))
    # Count loop inside the fold: one new member and one score per count.
    for position, value in enumerate(forest_n_estimators_values):
        started = time.perf_counter()
        growing.set_params(n_estimators=value).fit(fold["X_fit"], fold["y_fit"])
        seconds[position] += time.perf_counter() - started
        fit_sum += growing.estimators_[-1].predict_proba(fold["X_fit"])
        score_sum += growing.estimators_[-1].predict_proba(fold["X_score"])
        # Score the ensemble at this count from the running sums.
        fit_scores[position, fold_id] = f1_score(fold["y_fit"], fit_sum.argmax(axis=1))
        validation_scores[position, fold_id] = f1_score(fold["y_score"], score_sum.argmax(axis=1))
# A second pass on all the training rows, prepared once in Step 8, gives the test score and the out-of-bag error at every count; the test rows only record a score and never choose a count.
growing = clone(default_forest).set_params(warm_start=True, oob_score=True)
# Running sum of the member answers on the test rows.
test_sum = np.zeros((len(y_test), 2))
test_scores, oob_error, oob_f1_error, no_oob = [], [], [], []
covered = np.zeros(len(y_train), dtype=bool)
# Count loop on all the training rows.
for position, value in tqdm(enumerate(forest_n_estimators_values), desc='position, value', leave=True, position=0):
    started = time.perf_counter()
    growing.set_params(n_estimators=value).fit(Z_train, y_train)
    seconds[position] += time.perf_counter() - started
    # The rows that at least one tree omitted so far.
    covered |= np.bincount(growing.estimators_samples_[-1], minlength=len(y_train)) == 0
    test_sum += growing.estimators_[-1].predict_proba(Z_test)
    test_scores.append(f1_score(y_test, test_sum.argmax(axis=1)))
    oob_error.append(1 - growing.oob_score_)
    oob_f1_error.append(1 - f1_score(y_train[covered], growing.oob_decision_function_[covered].argmax(axis=1)))
    no_oob.append(int((~covered).sum()))
# Time to reach each count.
seconds = np.cumsum(seconds)
# Assemble the sweep table, one row per count.
for position, value in enumerate(forest_n_estimators_values):
    model = clone(default_forest).set_params(n_estimators=value)
    forest_n_estimators_rows.append({
        "configuration": " ".join(str(model).split()), "setting": str(value),
        "train": list(fit_scores[position]), "validation": list(validation_scores[position]),
        "train_F1": fit_scores[position].mean(), "cv_F1": validation_scores[position].mean(),
        "test_F1": test_scores[position],
        "oob_error": oob_error[position], "oob_F1_error": oob_f1_error[position], "no_oob_rows": no_oob[position],
        "seconds": seconds[position]})
forest_n_estimators_sweep = pd.DataFrame(forest_n_estimators_rows)
# Show selected counts and the best one; the figure below draws all 150.
shown = sorted({1, 2, 5, 10, 25, 50, 100, 150, int(forest_n_estimators_sweep.loc[forest_n_estimators_sweep["cv_F1"].idxmax(), "setting"])})
forest_n_estimators_sweep.set_index("setting").loc[[str(count) for count in shown], ["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_n_estimators_sweep.loc[forest_n_estimators_sweep["cv_F1"].idxmax()]
best_model = clone(default_forest).set_params(n_estimators=int(best["setting"])).fit(Z_train, y_train)
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best_model},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 39:__ Plot training and validation F1 beside the out-of-bag errors against the number of forest members.

In [ ]:
# One panel for the training and validation scores and one panel per out-of-bag error, against the number of members.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(forest_n_estimators_values, forest_n_estimators_sweep["train_F1"], linewidth=2, color=PLOT_ORANGE, label="training (ten folds)")
axes[0].plot(forest_n_estimators_values, forest_n_estimators_sweep["cv_F1"], linewidth=2, color=PLOT_BLUE, label="validation (ten folds)")
axes[1].plot(forest_n_estimators_values, forest_n_estimators_sweep["oob_error"], linewidth=2, color=PLOT_BLUE, label="1 - OOB accuracy")
axes[1].plot(forest_n_estimators_values, forest_n_estimators_sweep["oob_F1_error"], linewidth=2, linestyle="--", color=PLOT_ORANGE, label="1 - OOB F1")
axes[0].set(xlabel="number of members", ylabel="F1", title="Training and validation F1")
axes[1].set(xlabel="number of members", ylabel="out-of-bag error", title="Out-of-bag error")
# Name the lines of every panel.
for axis in axes:
    axis.legend()
plt.tight_layout()
plt.show()

**Validation F1 rises from 0.7811 with one tree to 0.8457 at 10 and 0.8650 at 50, and the best of the 150 counts is 127 trees at 0.8695.** The default 100 trees score 0.8672 and 150 trees 0.8687, so the curve is flat above 50, and the forest at 100 trees scores above the bagged trees' 0.8611 at 100 members. 1 - OOB accuracy falls from 0.4794 with one tree to 0.2284 at 10 and 0.1656 at 150, and 1 - OOB F1 from 0.2236 to 0.1875 and 0.1309.

__Step 40:__ Change only forest `max_depth` on the fixed training folds, with the sweep list `[3, 6, 10, None]`.

### `forest: max_depth`

`max_depth` bounds the longest path from the root of each tree to a leaf; its default is `None`, which leaves the other stopping rules to decide. A lower bound makes each tree simpler, so the fit to the training rows loosens, the trees agree more and each fit is cheaper; a higher bound lets each tree follow the training rows more closely, which lowers its bias and raises its variance, a variance the average over trees then reduces.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_max_depth_values = ...
forest_max_depth_rows = []
for value in tqdm(forest_max_depth_values, desc='forest max depth values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(max_depth=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_max_depth_rows.append(row)
# The sweep table, one row per tested value.
forest_max_depth_sweep = pd.DataFrame(forest_max_depth_rows)
forest_max_depth_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_max_depth_sweep.loc[forest_max_depth_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8276 at depth 3, 0.8581 at 6, 0.8654 at 10 and 0.8672 unbounded, with training F1 0.8351, 0.8778, 0.9389 and 1.0000.** Every cut in depth costs validation score, and both out-of-bag errors fall as the depth grows, 1 - OOB accuracy 0.2119, 0.1828, 0.1706 and 0.1672.

__Step 41:__ Change only forest `min_samples_leaf` on the fixed training folds, with the sweep list `[1, 2, 5, 10, 20]`.

### `forest: min_samples_leaf`

`min_samples_leaf` is the smallest number of training rows a leaf may hold; its default is `1`. Raising it stops splits that would isolate a few rows, which smooths the trees, loosens the fit to the training rows and lowers the cost of growing them. Lowering it to the default lets each tree memorise single rows and leaves the averaging to remove the variance.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_min_samples_leaf_values = [1, 2, 5, 10, 20]
forest_min_samples_leaf_rows = []
for value in tqdm(forest_min_samples_leaf_values, desc='forest min samples leaf values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(min_samples_leaf=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_min_samples_leaf_rows.append(row)
# The sweep table, one row per tested value.
forest_min_samples_leaf_sweep = pd.DataFrame(forest_min_samples_leaf_rows)
forest_min_samples_leaf_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_min_samples_leaf_sweep.loc[forest_min_samples_leaf_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8672 at one row per leaf, 0.8683 at 2, 0.8635 at 5, 0.8605 at 10 and 0.8581 at 20.** Training F1 falls from 1.0000 to 0.8748 over the list, and the best validation score sits at 2 rows. The out-of-bag errors are lowest at 5 rows, 0.1666 and 0.1312, so they prefer a slightly larger leaf than validation does.

__Step 42:__ Change only forest `max_features` on the fixed training folds, with the sweep list `['sqrt', 'log2', 0.3, 0.5, 1.0]`.

### `forest: max_features`

`max_features` is the number or fraction of candidate columns each split may consider, drawn anew at every split; its default is `'sqrt'`, the square root of the columns. Fewer candidates make the trees differ more from one another, which lowers the variance of their average, but each split is chosen from less and may be worse. Offering every column at every split leaves only the bootstrap rows to make the trees differ.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_max_features_values = ['sqrt', 'log2', 0.3, 0.5, 1.0]
forest_max_features_rows = []
for value in tqdm(forest_max_features_values, desc='forest max features values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(max_features=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_max_features_rows.append(row)
# The sweep table, one row per tested value.
forest_max_features_sweep = pd.DataFrame(forest_max_features_rows)
forest_max_features_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_max_features_sweep.loc[forest_max_features_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 is 0.8672 for the square root of the columns, 0.8662 for the base-2 logarithm, 0.8670 at 30 percent, 0.8657 at 50 and 0.8606 for all of them.** The square root wins by a small margin, and all columns scores lowest. The out-of-bag errors are lowest at 30 percent, 0.1659 and 0.1316, below the square root's 0.1672 and 0.1322, and highest at 50 percent.

__Step 43:__ Change only forest `bootstrap` on the fixed training folds, with the sweep list `[False, True]`.

### `forest: bootstrap`

`bootstrap` decides whether each tree is grown on a draw of the rows with replacement; its default is `True`. With replacement each tree omits some rows, which makes the trees differ and leaves out-of-bag rows to score. Without replacement every tree is grown on all the rows, so only the column sampling at each split can make the trees differ, and no row is out of bag.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_bootstrap_values = [False, True]
forest_bootstrap_rows = []
for value in tqdm(forest_bootstrap_values, desc='forest bootstrap values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(bootstrap=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_bootstrap_rows.append(row)
# The sweep table, one row per tested value.
forest_bootstrap_sweep = pd.DataFrame(forest_bootstrap_rows)
forest_bootstrap_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_bootstrap_sweep.loc[forest_bootstrap_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Without replacement the forest scores 0.8647 and with replacement 0.8672.** The forest's per-split column sampling keeps its trees different even when every tree sees every row, which the bagging wrapper's 0.7971 above did not; the full-size draw without replacement omits no rows, so the out-of-bag columns hold NaN.

__Step 44:__ Change only forest `max_samples` on the fixed training folds, with the sweep list `[0.3, 0.5, 0.7, 1.0]`.

### `forest: max_samples`

`max_samples` is the number or fraction of rows each tree draws when `bootstrap=True`; its default is `None`, a draw as large as the training rows. A smaller draw makes each tree see less data and differ more from the others, which lowers the variance of the average at the price of a looser fit for each tree, and it makes each tree cheaper to grow. It has no effect without a bootstrap, and scikit-learn requires `None` in that case.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_max_samples_values = [0.3, 0.5, 0.7, 1.0]
forest_max_samples_rows = []
for value in tqdm(forest_max_samples_values, desc='forest max samples values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(max_samples=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_max_samples_rows.append(row)
# The sweep table, one row per tested value.
forest_max_samples_sweep = pd.DataFrame(forest_max_samples_rows)
forest_max_samples_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_max_samples_sweep.loc[forest_max_samples_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Validation F1 climbs from 0.8636 at 30 percent of the rows to 0.8672 for the full draw, with 0.8661 at 50 and 0.8666 at 70.** Training F1 climbs from 0.9329 to 1.0000. The out-of-bag errors are lowest at 70 percent, 0.1653 and 0.1302.

__Step 45:__ Change only forest `criterion` on the fixed training folds, with the sweep list `['gini', 'entropy', 'log_loss']`.

### `forest: criterion`

`criterion` is the rule that scores a candidate split; its default is `'gini'`. `'gini'` and `'entropy'` score a split by how mixed the classes stay in the child nodes, and `'log_loss'` is the same quantity as `'entropy'`; changing the criterion changes which split a node picks, rarely how deep the tree grows.

In [ ]:
# The values tested one at a time, with every other setting at its default.
forest_criterion_values = ['gini', 'entropy', 'log_loss']
forest_criterion_rows = []
for value in tqdm(forest_criterion_values, desc='forest criterion values', leave=True, position=0):
    # A copy of the default ensemble with only this setting changed.
    model = clone(default_forest).set_params(criterion=value)
    started = time.perf_counter()
    # One ensemble per prepared training fold; their scores on the fitted and on the held-out rows give the train and validation columns.
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(prepared, desc="prepared folds", unit="fold", leave=False)]
    # The fit on the all-rows Z_train, prepared once in Step 8, also scores its out-of-bag rows, which only a bootstrap leaves out; this fit is the model the log keeps.
    oob_model = clone(model).set_params(oob_score=model.bootstrap)
    fitted = clone(oob_model).fit(Z_train, y_train)
    # One table row: the setting, the ten fold scores and the all-rows fit.
    row = {"configuration": " ".join(str(model).split()), "setting": str(value),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(prepared, fold_models)],
           "model": fitted}
    # The table reports means over the ten folds; the test score is recorded beside them and never chooses a value.
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["seconds"] = time.perf_counter() - started
    # Out-of-bag error: each training row is scored by the members that did not draw it, and rows no member omitted are left out.
    if model.bootstrap:
        ensemble = fitted
        omitted = sum(np.bincount(rows, minlength=len(y_train)) == 0 for rows in ensemble.estimators_samples_)
        scored = omitted > 0
        row["oob_error"] = 1 - ensemble.oob_score_
        row["oob_F1_error"] = 1 - f1_score(y_train[scored], ensemble.oob_decision_function_[scored].argmax(axis=1))
        row["no_oob_rows"] = int((~scored).sum())
    else:
        row["oob_error"] = row["oob_F1_error"] = row["no_oob_rows"] = np.nan
    forest_criterion_rows.append(row)
# The sweep table, one row per tested value.
forest_criterion_sweep = pd.DataFrame(forest_criterion_rows)
forest_criterion_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "oob_error", "oob_F1_error", "no_oob_rows", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = forest_criterion_sweep.loc[forest_criterion_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train),
    "folds": folds.get_n_splits(), "name": best["configuration"],
    "train": best["train"], "validation": best["validation"], "model": best["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Gini scores 0.8672, and entropy and log loss both score 0.8684, with the same training F1 1.0 and the same test F1 0.8746.** The two rows are identical, so the two names behave as one criterion here, and their out-of-bag errors, 0.1716 and 0.1361, are higher than gini's 0.1672 and 0.1322.

__Step 46:__ Plot training and validation F1 and the out-of-bag errors for each of the other forest settings, one row of panels per setting.

In [ ]:
# The sweep tables of the other settings, one row of panels per setting.
forest_sweeps = {"max_depth": forest_max_depth_sweep, "min_samples_leaf": forest_min_samples_leaf_sweep, "max_features": forest_max_features_sweep, "bootstrap": forest_bootstrap_sweep, "max_samples": forest_max_samples_sweep, "criterion": forest_criterion_sweep}
# Numeric settings get connecting lines, categories get separate points.
ordered = ("max_depth", "min_samples_leaf", "max_samples")
# Left: training and validation score; the other panels: the out-of-bag errors.
fig, axes = plt.subplots(len(forest_sweeps), 2, figsize=(12, 3.2 * len(forest_sweeps)))
# One row of panels per setting, drawn from its sweep table.
for (setting, sweep), (score_axis, oob_axis) in zip(forest_sweeps.items(), axes):
    line = "-" if setting in ordered else "none"
    # Left panel: the fold-mean training and validation scores for each value of this setting.
    score_axis.plot(sweep["setting"], sweep["train_F1"], marker="o", markersize=8, linestyle=line, color=PLOT_ORANGE, label="training (ten folds)")
    score_axis.plot(sweep["setting"], sweep["cv_F1"], marker="o", markersize=8, linestyle=line, color=PLOT_BLUE, label="validation (ten folds)")
    # Other panels: the out-of-bag errors; a value without a bootstrap has none and leaves a gap.
    oob_axis.plot(sweep["setting"], sweep["oob_error"], marker="o", markersize=8, linestyle=line, color=PLOT_BLUE, label="1 - OOB accuracy")
    oob_axis.plot(sweep["setting"], sweep["oob_F1_error"], marker="s", markersize=8, linestyle=line, color=PLOT_ORANGE, label="1 - OOB F1")
    # Titles and axis names carry the setting.
    score_axis.set(xlabel=setting, ylabel="F1", title=f"{setting}: training and validation F1")
    oob_axis.set(xlabel=setting, ylabel="out-of-bag error", title=f"{setting}: out-of-bag error")
    score_axis.legend(fontsize=8)
    oob_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

**The out-of-bag panels pick the same depth as the validation panels and a different value for leaf size, `max_features`, `max_samples` and criterion.** For `max_depth` both are best for unbounded trees, 0.8672 for validation and 0.1672 and 0.1322 out of bag. For `min_samples_leaf` validation is best at 2 rows, 0.8683, and the out-of-bag errors at 5, 0.1666 and 0.1312. For `max_features` validation is best at the square root, 0.8672, and the out-of-bag errors at 30 percent, 0.1659 and 0.1316. For `max_samples` validation is best for the full draw, 0.8672, and the out-of-bag errors at 70 percent, 0.1653 and 0.1302. For the criterion validation prefers entropy, 0.8684, and the out-of-bag errors gini, 0.1672 and 0.1322. The differences between the values of each of these settings are small on both measures, and the `bootstrap` panels hold one out-of-bag point because the full-size draw without replacement omits no rows.

__Step 47:__ Select the best tested forest configuration from the existing sweep rows by validation score.

In [ ]:
forest_tested = pd.concat([forest_n_estimators_sweep, forest_max_depth_sweep, forest_min_samples_leaf_sweep, forest_max_features_sweep, forest_bootstrap_sweep, forest_max_samples_sweep, forest_criterion_sweep], ignore_index=True)
best_tested_forest = forest_tested.loc[forest_tested["cv_F1"].idxmax()]
print("best tested forest configuration:", best_tested_forest["configuration"])
print(best_tested_forest[["train_F1", "cv_F1", "test_F1"]])

**The best tested forest configuration is 127 trees, at validation F1 0.8695 and test F1 0.8774.** Its training F1 is 1.0000, and it is the `n_estimators` sweep row, ahead of the 0.8684 of the entropy criterion and the 0.8683 at two rows per leaf.

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

A joint search chooses settings on inner training folds. The outer folds of the training partition then score that whole choice on rows the search never saw, preprocessing included. The final grid is fitted only after this estimate exists.

The search uses prespecified conditional grids, written before any search score exists. Bagging crosses all five taught learners with every member count from 1 to 150 and with every sampling setting, and forests cross every member count from 1 to 150, depth, leaf size, `max_features`, criterion and the bootstrap branches, where `max_samples` applies only with `bootstrap=True`. The value lists are the lists the one-setting sweeps tested, read from those cells, so the search can combine any values the sweeps showed.

__Step 48:__ Declare the conditional grids, which fix the search space before any search score exists.

In [ ]:
# The search space reuses every value the one-setting sweeps tested; the lists are read from the sweep cells above, not retyped.
# Each group is one family, and the forest has two groups because max_samples only applies when bootstrap=True.
joint_grid = [
    # Bagging group: each taught learner as the member.
    {"estimator": [BaggingClassifier(random_state=random_state, n_jobs=4)],
     "estimator__estimator": bag_estimator_values,
     "estimator__n_estimators": bag_n_estimators_values,
     "estimator__max_samples": bag_max_samples_values,
     "estimator__max_features": bag_max_features_values,
     "estimator__bootstrap": bag_bootstrap_values,
     "estimator__bootstrap_features": bag_bootstrap_features_values},
    # Forest without a bootstrap: rows are not drawn, so max_samples stays None.
    {"estimator": [RandomForestClassifier(random_state=random_state, n_jobs=4)],
     "estimator__n_estimators": forest_n_estimators_values,
     "estimator__max_depth": forest_max_depth_values,
     "estimator__min_samples_leaf": forest_min_samples_leaf_values,
     "estimator__max_features": forest_max_features_values,
     "estimator__criterion": forest_criterion_values,
     "estimator__bootstrap": [value for value in forest_bootstrap_values if not value],
     "estimator__max_samples": [None]},
    # Forest with a bootstrap: max_samples sets the draw size.
    {"estimator": [RandomForestClassifier(random_state=random_state, n_jobs=4)],
     "estimator__n_estimators": forest_n_estimators_values,
     "estimator__max_depth": forest_max_depth_values,
     "estimator__min_samples_leaf": forest_min_samples_leaf_values,
     "estimator__max_features": forest_max_features_values,
     "estimator__criterion": forest_criterion_values,
     "estimator__bootstrap": [value for value in forest_bootstrap_values if value],
     "estimator__max_samples": forest_max_samples_values},
]
# The printed summary shows the member-count range and the learners the bagging group crosses.
print("prespecified member counts:", joint_grid[0]["estimator__n_estimators"][0], "to", joint_grid[0]["estimator__n_estimators"][-1])
print("bagging learners:", list(taught_learners))

__Step 49:__ Enumerate every valid combination, and draw 60 of them without replacement before any score is seen.

In [ ]:
space = [candidate for group in joint_grid for candidate in ParameterGrid(group)]
print("combinations by group:", [len(ParameterGrid(group)) for group in joint_grid])
print("combinations in the space:", len(space))
draw = np.random.RandomState(random_state).choice(len(space), size=min(60, len(space)), replace=False)
search_grid = [{name: [value] for name, value in space[position].items()} for position in sorted(draw)]
print("combinations evaluated:", len(search_grid))

**The conditional grids hold 285,000 valid combinations, 60,000 for bagging and 45,000 and 180,000 for the two forest branches, and the search evaluates 60 of them.**

The search evaluates 60 combinations drawn uniformly at random, the same 60 in every nested search and in the final fit. Sixty random draws reach the best 5 percent of a space with about 95 percent probability, whatever its size ([Bergstra and Bengio, 2012](https://jmlr.org/papers/v13/bergstra12a.html)). With all parameter values supplied as lists, scikit-learn 1.9's `RandomizedSearchCV` uses `ParameterSampler`, which samples uniformly without replacement over the combinations indexed by `ParameterGrid`. The explicit sampling here freezes the same 60 candidates for every nested search and the final fit.

__Step 50:__ Wrap the grid and the preprocessing in one search, with three inner folds.

In [ ]:
inner_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
search_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(recipe), default_bag), search_grid,
    scoring="f1", cv=inner_folds, refit=True, n_jobs=4)

__Step 51:__ Count the candidates and the model fits the search can cost.

In [ ]:
candidate_count = len(search_grid)
print("candidates:", candidate_count)
print("inner-fold candidate fits per search:", candidate_count * 3)
print("searches including final refit:", folds.get_n_splits() + 1)
# The largest ensemble among the drawn candidates sets the bound on the members one search can fit.
largest_ensemble = max(candidate["estimator__n_estimators"][0] for candidate in search_grid)
print("largest drawn ensemble:", largest_ensemble, "members")
print("member-fit upper bound including search winner refits:", (candidate_count * 3 * largest_ensemble + largest_ensemble) * (folds.get_n_splits() + 1))

The bound counts ensemble members, taking every candidate to be as large as the largest one drawn; calibrated base learners can perform additional internal fits.

__Step 52:__ Estimate the whole search with the ten outer folds of the training partition, run at the same time on ten workers; the scores do not depend on the worker count.

In [ ]:
# Time the whole nested estimate.
# The search selects on F1 alone; the two companion metrics only score the refitted winner of each outer fold.
nested_metrics = {"f1": "f1", "roc_auc": "roc_auc", "balanced_accuracy": "balanced_accuracy"}
nested_started = time.perf_counter()
# Each outer training fold runs the complete search on its own worker; the generator lets the bar advance as folds finish.
outer_runs = joblib.Parallel(n_jobs=10, return_as="generator")(
    joblib.delayed(cross_validate)(search_template, X_train, y_train, cv=[split],
        scoring=nested_metrics, return_train_score=True, n_jobs=1, error_score="raise")
    for split in folds.split(X_train, y_train))
# Collect the ten outer results in fold order.
fold_results = list(tqdm(outer_runs, total=folds.get_n_splits(),
                         desc="Nested CV: outer folds", unit="fold", leave=True, position=0))
# Stack the training and validation scores of the ten outer folds, one array per metric.
nested_scores = {key: np.concatenate([result[key] for result in fold_results])
                 for key in fold_results[0] if key.startswith(("train_", "test_"))}
nested_seconds = time.perf_counter() - nested_started
# The mean validation F1 is the nested estimate of the whole procedure; the search still selects on F1.
nested_validation = nested_scores["test_f1"]
nested_train = nested_scores["train_f1"]
# Report the estimate, its range over the folds and the time.
print(f"nested validation F1: {nested_validation.mean():.4f}")
print(f"fold range: {nested_validation.min():.4f} to {nested_validation.max():.4f}")
print(f"nested validation ROC-AUC: {nested_scores['test_roc_auc'].mean():.4f}")
print(f"nested validation balanced accuracy: {nested_scores['test_balanced_accuracy'].mean():.4f}")
print(f"nested search wall time: {nested_seconds:.2f} seconds")

**The nested validation F1 is 0.8661, with the ten outer folds ranging from 0.8440 to 0.8867.** The ten searches took 131.32 seconds, about two minutes.

The outer mean estimates the search procedure, selection step included. It is not the inner-CV score of whichever candidate later wins on all the training rows; both can be reported, and they answer different questions. [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)

# <font color='#E8800A'>The final grid and its winner</font> <a class="anchor" id="grid"></a>
[Back to TOC](#toc)

With the search space fixed and its nested estimate available, the same candidates are fitted to all the training rows. Its best estimator supplies the predictions, the model artifact and the course row, even when it differs from the default or from a sweep winner.

__Step 53:__ Run the declared joint grid on all the training rows and keep its winner.

In [ ]:
# Run the same search on all the training rows.
final_started = time.perf_counter()
final_search = clone(search_template).fit(X_train, y_train)
final_seconds = time.perf_counter() - final_started
# The refitted best estimator and its settings feed the rest of the notebook.
chosen_model = final_search.best_estimator_
chosen_parameters = final_search.best_params_
# Report the chosen settings, the winner's own inner score and the time.
print("selected parameters:", chosen_parameters)
print("winner own inner-CV score:", final_search.best_score_)
print(f"final search wall time: {final_seconds:.2f} seconds")

**The search keeps a forest of 135 trees with the log-loss criterion, a depth limit of 10, two rows per leaf, the square root of the columns at each split and a bootstrap draw of the rows as large as the training rows, and its own inner F1 is 0.8660.** The search on all the training rows took 45.30 seconds.

__Step 54:__ Score the winner on the ten outer folds, and log it beside its test score.

In [ ]:
# Score the winner on the same folds: its settings are fitted on each prepared training part.
chosen_train, chosen_validation = [], []
for fold in tqdm(prepared, desc='prepared', leave=True, position=0):
    fitted = clone(chosen_model.estimator).fit(fold["X_fit"], fold["y_fit"])
    chosen_train.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
    chosen_validation.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))

In [ ]:
# The log names the winner by every setting the grid chose.
settings = final_search.best_params_
chosen_label = type(settings["estimator"]).__name__ + "(" + ", ".join(
    f"{name.split('__')[-1]}={value!r}" for name, value in settings.items() if name != "estimator") + ")"
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": chosen_label, "train": chosen_train, "validation": chosen_validation,
    "model": chosen_model}, X_test, y_test, "winner's own score", metric=f1_score, label="F1")

__Step 55:__ Log the winner with its nested estimate as the validation score.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": f"joint search over {len(final_search.cv_results_['params'])} combinations", "train": nested_train, "validation": nested_validation,
    "model": chosen_model}, X_test, y_test, "nested estimate", metric=f1_score, label="F1")

__Step 56:__ Read the winner's test score from the log.

In [ ]:
final_test = model_log[-1]["test F1"]
print(f"actual grid winner: test F1 {final_test:.4f}")

**The winner scores F1 0.8816 on the 800 held-out athletes, against 0.8661 for its nested estimate and 0.8651 for its own score over the ten outer folds.** The search chose the winner on the rows behind that own score, so the nested estimate is the one the board publishes.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The winner was chosen on the training folds. The board places its nested estimate and its test score beside the winners of the earlier weeks and the floor.

__Step 57:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor labels every athlete with the majority outcome.
results_dir = "../../results"
# The board knows the data, recipe and rows, so only comparable earlier results join it.
board = Scoreboard("champions outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train)),
    floor=round(f1_score(y_test, DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)), 4))
# Add the results earlier weeks saved.
for week, earlier_model, earlier_cv, earlier_test in results_before(10, "classification", results_dir,
                                                                    metric=board.metric, context=board.context):
    board.record(week, earlier_model, earlier_cv, earlier_test)

__Step 58:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names every setting the grid chose.
board.record(10, chosen_label, round(nested_validation.mean(), 4), round(final_test, 4))
# F1 selects and publishes the row; ROC-AUC and balanced accuracy are companions measured on the same fitted models, with no extra fits.
winner_probability = chosen_model.predict_proba(X_test)[:, 1]
companion_metrics = pd.DataFrame({
    "nested outer-fold mean": {"F1": nested_validation.mean(), "ROC-AUC": nested_scores["test_roc_auc"].mean(),
                               "balanced accuracy": nested_scores["test_balanced_accuracy"].mean()},
    "held-out winner": {"F1": final_test, "ROC-AUC": roc_auc_score(y_test, winner_probability),
                        "balanced accuracy": balanced_accuracy_score(y_test, chosen_model.predict(X_test))}}).round(4)
display(companion_metrics)
# Save this week's row, with the untuned baseline, for later boards.
board.save(
    results_dir, 10, "classification", baseline_cv=round(np.mean(default_validation), 4),
    baseline_test=round(model_log[0]["test F1"], 4),
)

# The fitted winner is saved beside its result. Loading needs preprocessing.py on the path.
joblib.dump(chosen_model, f"{results_dir}/week_10_classification_model.joblib")

# Show the board.
print(board.table())

<div class="alert alert-block alert-success">

**`RandomForestClassifier` with 135 trees, the log-loss criterion and a depth limit of 10 joins the board at F1 0.8661 over the ten outer folds and 0.8816 held out,** above the floor's 0.7529 and above every earlier week: logistic regression's 0.8486 and 0.8512, naive Bayes's 0.8055 and 0.8245, KNN's 0.8518 and 0.8615 and the tree's 0.8475 and 0.8402. The row and the fitted winner are saved in `results/`.

</div>

# <font color='#E8800A'>Creating two equal models</font> <a class="anchor" id="equal"></a>
[Back to TOC](#toc)

The search above treated bagging and a random forest as two families, yet a bagged tree ensemble and a forest can be exactly the same fitted model. Equality requires matching row draws, available columns, tree growth and aggregation. The two objects below are fitted to a small constructed problem, so equality is read from the tree arrays and not inferred from a rounded score.

The first column is ordered and the second is a fixed permutation of it. The response is a step function of the first column, which gives an unambiguous split at the class boundary. Both wrappers draw seven full-size bootstrap samples with every column available at each split, so the different tree random streams do not change the winning splits.

__Step 59:__ Fit a bagging wrapper and a random forest on the constructed problem, and compare their bootstrap draws.

In [ ]:
# A small constructed problem: an ordered column, a permuted copy of it and a step response.
equality_X = np.column_stack([np.arange(80), np.random.RandomState(random_state).permutation(80)])
equality_y = (equality_X[:, 0] >= 40).astype(int)
# The bagging wrapper with every column available and full-size bootstrap draws.
equal_bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3, max_features=1.0),
    n_estimators=7, max_samples=1.0, max_features=1.0,
    bootstrap=True, bootstrap_features=False, random_state=random_state, n_jobs=4).fit(equality_X, equality_y)
# The forest with the same draws and every column at each split.
equal_forest = RandomForestClassifier(max_depth=3, max_features=1.0,
    n_estimators=7, max_samples=1.0, bootstrap=True, random_state=random_state, n_jobs=4).fit(equality_X, equality_y)
# Two separate objects that draw the same rows.
print("separately fitted objects:", equal_bag is not equal_forest)
print("same bootstrap draws:", all(np.array_equal(a, b)
    for a, b in zip(equal_bag.estimators_samples_, equal_forest.estimators_samples_)))

__Step 60:__ Compare the two ensembles tree by tree: child links, split columns, thresholds, leaf values and weighted row counts.

In [ ]:
# Compare the two ensembles member by member.
tree_checks = []
for member_id, (bag_tree, forest_tree_pair) in enumerate(zip(equal_bag.estimators_, equal_forest.estimators_)):
    # Exact equality of each tree array.
    checks = {attribute: np.array_equal(getattr(bag_tree.tree_, attribute), getattr(forest_tree_pair.tree_, attribute))
        for attribute in ("children_left", "children_right", "feature", "threshold", "value", "weighted_n_node_samples")}
    # The largest numeric differences of thresholds and values.
    checks["threshold difference"] = np.max(np.abs(bag_tree.tree_.threshold - forest_tree_pair.tree_.threshold))
    checks["value difference"] = np.max(np.abs(bag_tree.tree_.value - forest_tree_pair.tree_.value))
    checks["member"] = member_id
    tree_checks.append(checks)
# One row per member.
equality_table = pd.DataFrame(tree_checks).set_index("member")
display(equality_table)
# Every array must match, and the bootstrap draws must be the same.
assert equality_table[["children_left", "children_right", "feature", "threshold", "value", "weighted_n_node_samples"]].all().all()
assert all(np.array_equal(a, b) for a, b in zip(equal_bag.estimators_samples_, equal_forest.estimators_samples_))
print("matching trees:", len(equality_table), "of", len(equal_bag.estimators_))

__Step 61:__ Compare the two aggregated answers on 161 probe rows that no tree was fitted on.

In [ ]:
# Compare aggregate probabilities on new rows after aligning the class order.
assert np.array_equal(equal_bag.classes_, equal_forest.classes_)
equality_probe = np.column_stack([np.linspace(0, 79, 161), np.linspace(79, 0, 161)])
bag_answers = equal_bag.predict_proba(equality_probe)
forest_answers = equal_forest.predict_proba(equality_probe)
assert np.array_equal(bag_answers, forest_answers)
print("probe rows:", len(equality_probe))
print("exact aggregated probabilities:", np.array_equal(bag_answers, forest_answers))
print("largest aggregate difference:", np.max(np.abs(bag_answers - forest_answers)))

Every child link, split column, threshold, leaf value and weighted row count must match exactly in the table. The two estimators still have different Python classes and internal tree seeds.

**All seven pairs of trees match on every array, with threshold and value differences of 0.0, and the 161 probe rows receive identical probabilities.** The two wrappers define the same fitted prediction function in this construction.

__Step 62:__ Change one thing in each wrapper: one candidate column per split in the forest, one column per member in the bagging wrapper.

In [ ]:
subsampled_forest = RandomForestClassifier(n_estimators=7, max_depth=3, max_features=1,
    bootstrap=True, random_state=random_state, n_jobs=4).fit(equality_X, equality_y)
subsampled_bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=7,
    max_features=1, bootstrap=True, random_state=random_state, n_jobs=4).fit(equality_X, equality_y)
print("per-node forest changed probabilities:", not np.array_equal(equal_forest.predict_proba(equality_probe), subsampled_forest.predict_proba(equality_probe)))
print("per-member bagging changed probabilities:", not np.array_equal(equal_bag.predict_proba(equality_probe), subsampled_bag.predict_proba(equality_probe)))
print("bagging member feature indices:", [columns.tolist() for columns in subsampled_bag.estimators_features_])

The integer `max_features=1` means one column; the float `1.0` means all columns. Sampling one candidate column at each forest split is a different construction, and so is sampling one column per bagging member. [Tree feature sampling](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

**Sampling one candidate column at each forest split changed the probabilities, and so did giving each bagging member one column.** The bagging members drew the single columns [1], [1], [0], [0], [1], [0] and [0].

The same integer seed is **not an identity guarantee**: the two classes can consume random streams differently when they seed their trees, even when the bootstrap draws coincide, so on data with competing splits identical seeds need not select identical trees. [Bagging implementation](https://github.com/scikit-learn/scikit-learn/blob/1.9.0/sklearn/ensemble/_bagging.py), [forest implementation](https://github.com/scikit-learn/scikit-learn/blob/1.9.0/sklearn/ensemble/_forest.py)

__Step 63:__ Draw the first member of each constructed ensemble.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_tree(equal_bag.estimators_[0], feature_names=["ordered", "permuted"], filled=True, rounded=True, precision=2, ax=axes[0], fontsize=9)
plot_tree(equal_forest.estimators_[0], feature_names=["ordered", "permuted"], filled=True, rounded=True, precision=2, ax=axes[1], fontsize=9)
axes[0].set_title("Bagging: first fitted member")
axes[1].set_title("Random forest: first fitted member")
plt.tight_layout()
plt.show()

**The two first members are the same tree.** The root reports `samples = 52` beside `value = [38, 42]`, which adds to 80: `samples` counts the distinct rows of the draw, while `value` counts bootstrap draws, so a row drawn twice adds 2 to a class count and 1 to `samples`.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **Averaging bootstrap fits helps an unstable learner and leaves a stable one alone:** the deep tree moves from F1 0.7996 to 0.8374 when bagged.
2. **A full-size bootstrap leaves a share of the rows out of every member:** on average 0.3650 of the rows.
3. **More members help and then flatten:** validation F1 is 0.7872 with one member, 0.8374 at 10 and 0.8635 at 150.
4. **A forest at 100 trees scores F1 0.8672 against 0.8611 for 100 bagged trees, and the two can be the same model:** with every column at each split and the same bootstrap draws, all seven pairs of trees matched exactly, while sampling columns per split or per member changed the probabilities.
5. **The bootstrap gives an error estimate without holding rows back:** at 150 members 1 - OOB accuracy is 0.1750 and 1 - OOB F1 is 0.1385, read from the rows the members were fitted on.
6. **The nested estimate scores the search procedure:** the winner scores F1 0.8661 nested, 0.8660 on its own inner folds and 0.8816 held out.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

The bagged trees score validation F1 0.7872 with one member and 0.8607 with 50. In one sentence, why does averaging more trees score higher on athletes they were not fitted on?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

The API references define the supported parameter combinations; the original papers explain the sampling constructions and tree ensemble design.

- [scikit-learn bagging guide: pasting, bagging, random subspaces and random patches](https://scikit-learn.org/stable/modules/ensemble.html#bagging-meta-estimator)
- [scikit-learn BaggingClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)
- [scikit-learn RandomForestClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [Breiman, Bagging Predictors (1996)](https://www.stat.berkeley.edu/~breiman/bagging.pdf)
- [Breiman, Random Forests (2001)](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf)
- [Ho, The Random Subspace Method for Constructing Decision Forests (1998)](https://doi.org/10.1109/34.709601)
- [Louppe and Geurts, Ensembles on Random Patches (2012)](https://orbi.uliege.be/handle/2268/130099)